In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, f_regression 
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor


## Task1

In [4]:
df = pd.read_csv('Insurance.csv')

In [5]:
print(df.shape)

(1338, 7)


In [6]:
print(df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [7]:
df_encoded = pd.get_dummies(df, drop_first = True)
X_ins= df_encoded.drop(columns= ['charges'])
y_ins = df_encoded['charges']



In [8]:
X_ins_train, X_ins_test, y_ins_train, y_ins_test = train_test_split(X_ins, y_ins, test_size= 0.2, random_state = 42)

## Task 2

In [9]:
selector = VarianceThreshold(threshold=0)
X_train_vt = selector.fit_transform(X_ins_train)
X_test_vt = selector.transform(X_ins_test)
selected = selector.get_support()
kept = X_ins_train.columns[selected]
X_train_vt = pd.DataFrame(X_train_vt, columns=kept)
X_test_vt = pd.DataFrame(X_test_vt, columns=kept)

print(selected)

[ True  True  True  True  True  True  True  True]


In [10]:
print(X_ins_train.columns[selected])


Index(['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest',
       'region_southeast', 'region_southwest'],
      dtype='object')


In [11]:
X_ins_train.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
560,46,19.95,2,False,False,True,False,False
1285,47,24.32,0,False,False,False,False,False
1142,52,24.86,0,False,False,False,True,False
969,39,34.32,5,False,False,False,True,False
486,54,21.47,3,False,False,True,False,False


## Notes: Features have high variance as seen when going from .01 to 1.

## Task 3

In [12]:
skb = SelectKBest(score_func=f_regression, k = 'all' )
X_train_skb = skb.fit_transform(X_train_vt, y_ins_train)
X_ins_test = skb.transform(X_test_vt)

print(X_train_skb)

[[46.    19.95   2.    ...  1.     0.     0.   ]
 [47.    24.32   0.    ...  0.     0.     0.   ]
 [52.    24.86   0.    ...  0.     1.     0.   ]
 ...
 [58.    25.175  0.    ...  0.     0.     0.   ]
 [37.    47.6    2.    ...  0.     0.     1.   ]
 [55.    29.9    0.    ...  0.     0.     1.   ]]


In [13]:
scores = pd.Series(skb.scores_, index=X_train_vt.columns)
print(scores.sort_values(ascending=False))

smoker_yes          1659.952101
age                   92.070905
bmi                   43.265710
children               5.547503
region_southeast       4.887087
sex_male               3.457075
region_southwest       1.309463
region_northwest       1.219783
dtype: float64


1. Smoker_yes had the highest score.
2. Yes it does match because smokers usually have higher insurance cost.
3. High scores mean they correlate with the charge cost.

In [14]:
#Task 4

lasso = Lasso(alpha=100, random_state=42)
lasso.fit(X_ins_train, y_ins_train)
print(pd.Series(lasso.coef_, index=X_ins.columns))

age                   256.128819
bmi                   324.835277
children              362.948153
sex_male                0.000000
smoker_yes          23041.824357
region_northwest        0.000000
region_southeast       -0.000000
region_southwest       -0.000000
dtype: float64


When lasso gives a column a coefficient of zero, it means that the column was dropped.
This is different from SelectKBest because SelectKBest only ranks the categories while lasso drops them.

In [ ]:
#Task 5

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_ins_train, y_ins_train)

importances = pd.Series(rf.feature_importances_, index = X_ins.columns).sort_values(ascending=False)
print(importances)

#TODO: Create a bar chart showing feature importance scores from highest to lowest. Label your axes and add a title.

smoker_yes          0.608618
bmi                 0.216506
age                 0.134232
children            0.019413
sex_male            0.006379
region_northwest    0.005587
region_southeast    0.005314
region_southwest    0.003950
dtype: float64
